In [0]:
WITH date_bounds AS (
    SELECT
        MIN(valid_from) AS earliest,
        MAX(valid_to)   AS latest
    FROM osv.silver.v_pypi_history
    WHERE valid_to != '9999-12-31'
),
changing_keys AS (
    SELECT DISTINCT id
    FROM osv.silver.v_pypi_history
    WHERE valid_to != '9999-12-31'
)
SELECT
    v.valid_from,
    v.valid_to,
    v.id,
    v.summary,
    v.modified,
    v.withdrawn,
    v.current_flag
FROM osv.silver.v_pypi_history v
WHERE v.id IN (SELECT id FROM changing_keys)
AND v.valid_from BETWEEN (SELECT earliest FROM date_bounds) AND (SELECT latest FROM date_bounds)
ORDER BY v.id, v.valid_from;

In [0]:
WITH date_bounds AS (
    SELECT
        MIN(valid_from) AS earliest,
        MAX(valid_to)   AS latest
    FROM osv.silver.v_pypi_history
    WHERE valid_to != '9999-12-31'
),
changing_keys AS (
    SELECT DISTINCT id
    FROM osv.silver.v_pypi_history
    WHERE valid_to != '9999-12-31'
),
ids_with_multiple_records AS (
    SELECT id
    FROM osv.silver.v_pypi_history
    WHERE id IN (SELECT id FROM changing_keys)
    GROUP BY id
    HAVING COUNT(*) > 2
)
SELECT
    v.valid_from,
    v.valid_to,
    v.id,
    v.summary,
    v.modified,
    v.withdrawn,
    v.current_flag
FROM osv.silver.v_pypi_history v
WHERE v.id IN (SELECT id FROM ids_with_multiple_records)
AND v.valid_from BETWEEN (SELECT earliest FROM date_bounds) AND (SELECT latest FROM date_bounds)
ORDER BY v.id, v.valid_from;

In [0]:
%python

dbutils.widgets.text('vuln_id', '')
dbutils.widgets.dropdown('ecosystem', 'pypi', ['pypi', 'cran'])

vuln_id = dbutils.widgets.get('vuln_id')
ecosystem = dbutils.widgets.get('ecosystem')

view_name = f'osv.silver.v_{ecosystem}_history'

latest_valid = spark.sql(f"SELECT MAX(valid_to) AS latest FROM {view_name} WHERE valid_to != '9999-12-31'").collect()[0]['latest']

query = f"""
SELECT
    valid_from,
    CASE
        WHEN valid_to = '9999-12-31' THEN DATE_ADD(DATE('{latest_valid}'), 90)
        ELSE valid_to
    END AS valid_to,
    CASE
        WHEN valid_to = '9999-12-31' THEN TRUE
        ELSE FALSE
    END AS is_current,
    summary,
    withdrawn,
    modified
FROM {view_name}
WHERE id = '{vuln_id}'
ORDER BY valid_from
"""

df = spark.sql(query).toPandas()

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

df['valid_from'] = pd.to_datetime(df['valid_from'])
df['valid_to'] = pd.to_datetime(df['valid_to'])

fig, ax = plt.subplots(figsize=(12, max(len(df) * 0.8, 2)))

for i, row in df.iterrows():
    color = 'salmon' if pd.notna(row['withdrawn']) and row['withdrawn'] else 'steelblue'
    
    ax.barh(
        y=i,
        width=(row['valid_to'] - row['valid_from']).days,
        left=row['valid_from'],
        height=0.8,
        color=color
    )
    
    # Add continuation indicator for current versions
    if row['is_current']:
        ax.text(
            row['valid_to'] - pd.Timedelta(days=3), i, '→∞',
            va='center', ha='right', fontsize=12, fontweight='bold',
            color='white'
        )

labels = []
for i, row in df.iterrows():
    w = ' [WITHDRAWN]' if pd.notna(row['withdrawn']) and row['withdrawn'] else ''
    version = f"v{i+1}"
    modified_date = pd.to_datetime(row['modified']).strftime('%Y-%m-%d') if pd.notna(row['modified']) else 'N/A'
    summary = (str(row['summary']) if pd.notna(row['summary']) else '')[:40]
    labels.append(f'{version} ({modified_date}): {summary}{w}')

ax.set_yticks(range(len(df)))
ax.set_yticklabels(labels)
ax.invert_yaxis()
ax.set_xlim(left=df['valid_from'].min(), right=df['valid_to'].max() + pd.Timedelta(days=5))
ax.set_xlabel('Date')
ax.set_title(f'{vuln_id} ({ecosystem})')

ax.xaxis.set_major_locator(mdates.AutoDateLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
fig.autofmt_xdate(rotation=0)

plt.tight_layout()
plt.show()

In [0]:
SELECT
    id,
    COUNT(*) AS version_count,
    MIN(valid_from) AS first_seen,
    MAX(CASE WHEN current_flag = 'Yes' THEN valid_from END) AS last_changed,
    MAX(CASE WHEN current_flag = 'Yes' THEN withdrawn END) AS current_withdrawn_date
FROM osv.silver.v_pypi_history
GROUP BY id
HAVING COUNT(*) > 1
ORDER BY version_count DESC
LIMIT 25;

In [0]:
-- Graph 1: Top 20 most vulnerable PyPI packages
-- Tip: horizontal bar chart, package on Y axis, vuln_count on X axis
-- TensorFlow variants (gpu/cpu/base) dominate; consider summing them or showing separately
SELECT
    aff.package.name AS package,
    COUNT(*) AS vuln_count
FROM osv.silver.v_pypi_history
LATERAL VIEW explode(affected) t AS aff
WHERE current_flag = 'Yes'
  AND aff.package.ecosystem = 'PyPI'
GROUP BY aff.package.name
ORDER BY vuln_count DESC
LIMIT 20

In [0]:
-- Graph 2: New vulnerabilities published per month
-- Note: Feb 2023 (~6200) and Jun 2024 (~1800) are bulk historical imports, not organic spikes
-- Tip: line chart with month on X axis, vuln_count on Y axis
SELECT
    DATE_TRUNC('month', published) AS month,
    COUNT(*) AS vuln_count,
    SLICE(COLLECT_LIST(id), 1, 5) AS sample_ids,
    SLICE(COLLECT_LIST(summary), 1, 5) AS sample_summaries
FROM osv.silver.v_pypi_history
WHERE current_flag = 'Yes'
  AND published IS NOT NULL
GROUP BY DATE_TRUNC('month', published)
ORDER BY month

In [0]:
-- Graph 3: Most cross-referenced vulnerabilities
-- High alias count = same vuln tracked by many databases = "notorious"
-- Tip: horizontal bar chart, id on Y axis, alias_count on X axis
SELECT
    id,
    COALESCE(summary, '(no summary)') AS summary,
    size(aliases) AS alias_count
FROM osv.silver.v_pypi_history
WHERE current_flag = 'Yes'
  AND aliases IS NOT NULL
  AND size(aliases) > 0
ORDER BY alias_count DESC
LIMIT 20

In [0]:
-- Graph 4: Alias type breakdown
-- Which external databases cross-reference PyPI vulnerabilities?
-- Tip: use a bar or pie chart on the 'count' column
SELECT
    REGEXP_EXTRACT(alias, '^([A-Z]+)', 1) AS alias_type,
    COUNT(*) AS count
FROM osv.silver.v_pypi_history
LATERAL VIEW explode(aliases) t AS alias
WHERE current_flag = 'Yes'
GROUP BY REGEXP_EXTRACT(alias, '^([A-Z]+)', 1)
ORDER BY count DESC

In [0]:
-- Graph 5b: How many vulnerabilities have 1, 2, or 3 CVEs assigned?
-- Tip: bar chart, cve_count on X axis, vuln_count on Y axis
SELECT
    SIZE(FILTER(aliases, a -> a LIKE 'CVE-%')) AS cve_count,
    COUNT(*) AS vuln_count
FROM osv.silver.v_pypi_history
WHERE current_flag = 'Yes'
  AND aliases IS NOT NULL
  AND SIZE(FILTER(aliases, a -> a LIKE 'CVE-%')) > 0
GROUP BY cve_count
ORDER BY cve_count

In [0]:
-- Graph 5: Vulnerabilities with the most CVEs assigned to them
-- More CVEs in one vuln = broader/more complex attack surface, tracked by more reporters
-- Tip: horizontal bar chart, id on Y axis, cve_count on X axis
SELECT
    id,
    COALESCE(summary, '(no summary)') AS summary,
    SIZE(FILTER(aliases, a -> a LIKE 'CVE-%')) AS cve_count,
    FILTER(aliases, a -> a LIKE 'CVE-%') AS cves
FROM osv.silver.v_pypi_history
WHERE current_flag = 'Yes'
  AND aliases IS NOT NULL
  AND SIZE(FILTER(aliases, a -> a LIKE 'CVE-%')) > 0
ORDER BY cve_count DESC
LIMIT 20